# 01 · HF 입력 준비

실행 로직은 프로젝트 모듈과 CLI에 있습니다. `CONFIG`의 훈련 step·평가 rollout·seed·학습률·주기·예산을 편집하거나 `OVERRIDES`로 지정하십시오. 재개는 같은 run_group과 고정 SOURCE_COMMIT 및 원래 설정을 사용합니다. 기본 실행 스위치는 모두 False입니다. 가중치/데이터는 고정 HF 입력이며 Drive는 자동 갱신하지 않습니다.

실험 단계: prepare_inputs → 훈련/검증 → select_teacher → cache_teacher → qwen_kd → evaluate_ava/evaluate_oft. Suite-balanced는 pilot4/main을 구분합니다.


In [ ]:
from pathlib import Path
import os, re, subprocess, sys, time
os.environ.setdefault('VLADE_SESSION_STARTED_UNIX', str(time.time()))
SOURCE_REF = 'main'  # 재개할 때는 이전 SOURCE_COMMIT을 지정하십시오.
REPOSITORY = 'https://github.com/eastha10/VLADE.git'
SOURCE_COMMIT = SOURCE_REF if re.fullmatch(r'[0-9a-f]{40}', SOURCE_REF) else subprocess.check_output(['git','ls-remote',REPOSITORY,SOURCE_REF], text=True).split()[0]
if not re.fullmatch(r'[0-9a-f]{40}', SOURCE_COMMIT): raise ValueError('불변 Git commit이 필요합니다.')
CODE = Path('/content/vlade_sources') / SOURCE_COMMIT
if any(name.startswith('vlade.') and getattr(module,'__file__',None) and not Path(module.__file__).resolve().is_relative_to(CODE) for name,module in list(sys.modules.items())):
    raise RuntimeError('다른 소스가 로드되어 있습니다. Colab 런타임을 다시 시작하십시오.')
if not CODE.exists():
    subprocess.run(['git','clone','--no-checkout','--filter=blob:none',REPOSITORY,str(CODE)],check=True)
    subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',SOURCE_COMMIT],check=True)
    subprocess.run(['git','-C',str(CODE),'checkout','--detach',SOURCE_COMMIT],check=True)
sys.path[:0] = [str(CODE/'src'),str(CODE)]
from vlade.utils.source_identity import source_identity
assert source_identity(CODE,require_clean=True)['git_commit']==SOURCE_COMMIT
print('SOURCE_COMMIT:',SOURCE_COMMIT)


In [ ]:
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')


In [ ]:
import datetime, uuid
from vlade.research.client import job
WORKSPACE = '/content/vlade_work'
CONFIG = str(CODE / 'configs/research/prepare_inputs.json')
RUN_GROUP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:8]
OVERRIDES = {'run_group': RUN_GROUP}  # 예: 'train.max_steps': 1000 또는 'evaluation.rollouts_per_task': 10
RUN_SETUP = False
RUN_EXPERIMENT = False


In [ ]:
job(CONFIG,action='plan',workspace=WORKSPACE,overrides=OVERRIDES)


In [ ]:
if RUN_SETUP:
    job(CONFIG,action='setup',workspace=WORKSPACE,overrides=OVERRIDES)


In [ ]:
if RUN_EXPERIMENT:
    job(CONFIG,action='run',workspace=WORKSPACE,overrides=OVERRIDES)


In [ ]:
RUN_FULLFRAMES = False  # 승인된 별도 공개 D/seed42 전체 프레임 게시
if RUN_FULLFRAMES:
    job(CODE/'configs/research/fullframes.json',action='setup',workspace=WORKSPACE)
    job(CODE/'configs/research/fullframes.json',action='run',workspace=WORKSPACE)
